# 04 · Test and select
**Brief: Task 3 ("Test each candidate and keep up to three").** Every candidate is scored with lift, precision
and coverage, with guardrails and normal-only thresholds; up to three survive. Code: `run_event_kind` in
`src/evaluation.py` §7 (protocol) and §6 (metrics).

### What this notebook answers from the brief
| Brief asks for | Where |
|---|---|
| **Lift**, **precision** and **coverage** for each detector | section 2 (every candidate, holdout machines) |
| Apply guardrails (e.g. exclude downtime or recovery periods) | section 1 (and `src/data.py` §3) |
| Set detector thresholds using normal operation only | section 1 (`src/detectors.py` §4) |
| Keep up to three of the strongest detectors | section 3 |
| Briefly explain why they beat the others | **section 4** |
| Test more than one H and W, and choose them | **section 6** (one F1 grid: 10 leads × 10 window lengths, live); section 7 re-tunes the rules for the deployment window |
| The event is configurable | section 8 (shortlist for every event type) |

In [1]:
# --- Setup ----------------------------------------------------------------------------
# Libraries, plus the project's helper files in ../src (data.py, detectors.py, evaluation.py).
# Every analysis function this notebook calls lives there, in numbered sections (§1 ... §9).
import json, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")  # harmless: kagglehub's progress bar

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

# Two outputs per chart: Plotly's own MIME type (rendered natively by VS Code and JupyterLab, works
# offline) plus an HTML fallback for classic Jupyter (loads plotly.js from a CDN).
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_colwidth", 120, "display.width", 200)  # show long rule texts in full

ROOT = Path.cwd().parent            # project root (this notebook lives in notebooks/)
sys.path.insert(0, str(ROOT / "src"))  # so `from data import ...` finds ../src/data.py

In [2]:
# --- Code used in this notebook (from ../src) --------------------------------------------
from data import (                            # src/data.py
    Config,                                   #   §1 settings: H, W, event, percentiles, seed
    load_panel, dev_holdout_split,            #   §2 raw data; which machines are dev / holdout
)
from detectors import FeatureBank             # src/detectors.py §4: features, cached
from evaluation import (                      # src/evaluation.py
    learn_mapping, run_event_kind, MAPPING,   #   §7 part<->signal/error pairing; the whole selection protocol
    f1_grid,                                  #   §7 dev F1 of the leading detector over a grid of H and W
    retune_for_window,                        #   §7 re-tune the kept rules for another window (deployment)
    save_results,                             #   §9 save this notebook's key results for notebooks 05 and 06
)

## 1 · The protocol, guardrails and thresholds
* **Guardrails** (`src/data.py` §3): the failure hour is never scored (it holds the repair record); the 24 h
  after any failure or repair are not "normal" and alerts there are muted; the first 24 h are warm-up; alerts
  in the last H+W hours are dropped (we can't see whether a failure followed).
* **Thresholds** (`src/detectors.py` §4): percentiles of **normal-operation hours only**, or robust per-machine
  statistics of those hours.
* **Protocol:** machines are split 50/50 into dev and holdout. Each idea keeps its best variant on dev, and must
  pass four gates on dev:
  1. **not luck:** beats a circular-shift null (each machine's alerts rotated in time at random), Bonferroni-corrected;
  2. **lift ≥ 2;**
  3. **coverage ≥ 5%;**
  4. **causal:** thresholds fitted on Jan–Jun alone keep ≥ half the Jul–Dec F1.

  Up to three are kept (dropping any that are the same alarm as a better one). Every number is reported on holdout.

> **Relies on:**  
> · `src/data.py` §3 — event windows and normal operation (`build_masks`)  
> · `src/evaluation.py` §6 — the metrics (`machine_stats`, `summarize`, `bootstrap`, `shift_null`)  
> · `src/evaluation.py` §7 — the selection protocol (`learn_mapping`, `run_event_kind`)

In [3]:
panel = load_panel()
bank = FeatureBank(panel)            # computes features (rolling means, error counts, ...) once and caches them
cfg = Config(event="any")            # <- change H, W or the event ("any", "comp1".."comp4") here

# Split MACHINES (not hours) 50/50 into dev and holdout, stratified by machine model.
# Every choice below is made on dev; holdout is only used to report. Splitting by machine means
# no machine helps choose a detector and then also grades it.
dev, hold = dev_holdout_split(panel, cfg.dev_fraction, cfg.seed)

# C09 ("matched signature") needs to know which signal and which error codes belong to which part.
# That pairing is learned on DEV machines only, so holdout labels never leak into it.
MAPPING.clear()
MAPPING.update(learn_mapping(panel, cfg, dev))
print("dev-learned pairing:", {c: (v["signal"], v["errors"]) for c, v in MAPPING.items()})

dev-learned pairing: {'comp1': ('volt', ['error1']), 'comp2': ('rotate', ['error2', 'error3']), 'comp3': ('pressure', ['error4']), 'comp4': ('vibration', ['error5'])}


**Conclusion.** The pairing learned on dev machines: comp1 → volt + error1, comp2 → rotate + error2/error3,
comp3 → pressure + error4, comp4 → vibration + error5.

## 2 · How we score "good"
| Metric | Question it answers | Gamed by |
|---|---|---|
| **Lift** | Does it fire more before failures than in normal operation? | a detector that almost never fires |
| **Precision** | When it raises an alert, how often does a failure follow within H+W? | firing only on the most obvious cases |
| **Coverage** | What share of failures had at least one alert in their window? | a detector that is always on |

Each can be gamed alone, so we select on **F1 of precision and coverage** (`2·P·C / (P + C)`): high only when
most failures are caught *and* most alarms are real. Alongside: the precision a random alarm would get, alerts
per machine per month, and lead time. Intervals are a **cluster bootstrap over holdout machines**.

In [4]:
# Run the whole protocol for "any failure" (src/evaluation.py §7): sweep every variant -> keep each idea's
# best variant on DEV -> null test, gates, causal check -> keep up to 3 -> report HOLDOUT.
# (About 1 minute; 500 shuffled-timing null draws per idea.)
res = run_event_kind(panel, cfg, bank, dev, hold, n_null=cfg.n_null_shifts)

# One row per idea. Sorted by dev F1, the score used for selection.
#   dev_f1 / hold_f1 : F1 on dev machines (used to choose) and on holdout machines (reported)
#   null_z           : how many standard deviations above the shuffled-timing null it scores (higher = less luck)
#   causal_f1        : Jul-Dec F1 when thresholds may use only Jan-Jun data (should stay close to dev)
#   reason           : why it was kept or rejected
tab = pd.DataFrame([dict(id=r["cid"], idea=r["title"], status=r["status"], variant=r["variant"],
                         dev_f1=r["dev"]["f1"], hold_f1=r["hold"]["f1"],
                         hold_lift=r["hold"]["lift"], hold_precision=r["hold"]["precision"],
                         hold_coverage=r["hold"]["coverage"], null_z=r["null_z"],
                         causal_f1=r["causal_f1"], reason=r["reason"]) for r in res["concepts"]])
tab.sort_values("dev_f1", ascending=False)

  [any H0_W48] 655 variants, kept ['C09', 'C08', 'C10'] (55s)


,id,idea,status,variant,dev_f1,hold_f1,hold_lift,hold_precision,hold_coverage,null_z,causal_f1,reason
8,C09,Matched signature (part's own signal + own error),kept,roll6·machine·p99·k1,0.9454,0.9178,337.8854,0.9271,0.9086,52.34,0.8989,rank 1 by dev F1 among detectors passing all gates
7,C08,Drift confirmed by an error,kept,roll6·robust·p99.99·k1,0.7767,0.7644,36.2531,0.6277,0.9771,53.29,0.7832,rank 2 by dev F1 among detectors passing all gates
9,C10,Drift on an aging part,kept,roll6·robust·p99.99·k1,0.5584,0.5663,16.4690,0.4126,0.9029,42.27,0.5748,rank 3 by dev F1 among detectors passing all gates
5,C06,Any error code,rejected,count24·global·p95·k1,0.5460,0.5664,81.2971,0.6436,0.5057,38.22,0.5605,"passed all gates but outranked by three stronger, non-redundant detectors"
4,C05,Any signal drifts,rejected,roll6·robust·p99.99·k1,0.4566,0.4412,9.0631,0.2835,0.9943,43.88,0.4512,"passed all gates but outranked by three stronger, non-redundant detectors"
1,C02,Rotation sags,rejected,roll6·robust·p99.99·k1,0.3870,0.3899,13.8289,0.3728,0.4086,29.40,0.3905,"passed all gates but outranked by three stronger, non-redundant detectors"
0,C01,Voltage creeps up,rejected,roll6·model·p99·k1,0.2990,0.2828,10.1494,0.2746,0.2914,21.71,0.2865,"passed all gates but outranked by three stronger, non-redundant detectors"
3,C04,Vibration rises,rejected,roll24·robust·p99.99·k1,0.2866,0.2881,7.7450,0.2848,0.2914,18.68,0.2597,"passed all gates but outranked by three stronger, non-redundant detectors"
2,C03,Pressure builds,rejected,raw·model·p99.5·k1,0.2356,0.2461,8.1375,0.2557,0.2371,15.05,0.2483,"passed all gates but outranked by three stronger, non-redundant detectors"
6,C07,Overdue maintenance,rejected,raw·machine·p99.99·k1,0.1353,0.1058,18.9547,0.4468,0.0600,14.48,0.0000,collapses with past-only thresholds (Jul–Dec F1 0.00 vs 0.05 with full-year thresholds): its skill leaks from the fu...


**Conclusion.** Dev and holdout F1 agree within ~0.02 for every idea, so nothing is overfit to the dev machines.
Nine of the ten ideas pass all four gates; only **C07 (overdue maintenance) fails**, on the causal check. The
three combinations lead: **C09 (F1 0.92), C08 (0.76), C10 (0.57)**. The best single idea, "any error code"
(C06), reaches 0.57.

## 3 · The shortlist, on holdout machines

In [5]:
# The kept detectors, measured on HOLDOUT machines only (never used to choose anything).
for r in res["concepts"]:
    if r["status"] != "kept":
        continue                                                     # rejected ideas are explained in section 4
    print(f"{r['cid']} {r['title']}\n   rule: {r['rule_short']}")   # the rule in compact plain words
    # The brief's three numbers, each with a 95% cluster-bootstrap interval (holdout machines resampled whole).
    for k in ("lift", "precision", "coverage"):
        lo, hi = r["hold_ci"][k]
        print(f"   {k:9s} {r['hold'][k]:.3f}   95% CI [{lo:.3f}, {hi:.3f}]")
    # What an operator would experience: how early the first alert comes, how many alerts per machine per
    # month, and the precision a randomly firing alarm would get (the bar this detector has to clear).
    print(f"   median lead {r['lead_median']} h | {r['hold']['alerts_per_machine_month']:.2f} alerts per machine-month"
          f" | random-alarm precision {r['hold']['base_precision']:.1%}\n")

C08 Drift confirmed by an error
   rule: (any of [volt ↑, rotate ↓, pressure ↑, vibration ↑]: 6h trailing mean beyond median ± 3.72×MAD-σ of its own machine's normal operation, in the arrow's direction) AND (error-log count in last 48h > P80 of normal operation (all machines))
   lift      36.253   95% CI [30.564, 43.234]
   precision 0.628   95% CI [0.584, 0.674]
   coverage  0.977   95% CI [0.958, 0.991]
   median lead 24.0 h | 2.19 alerts per machine-month | random-alarm precision 4.1%

C09 Matched signature (part's own signal + own error)
   rule: for each part [volt ↑, rotate ↓, pressure ↑, vibration ↑]: own signal 6h trailing mean beyond P99 (↑) / P1 (↓) of normal operation (its own machine) AND its own error code (error1 / error2+error3 / error4 / error5) count in last 24h > P80 of normal operation (all machines)
   lift      337.885   95% CI [230.453, 544.024]
   precision 0.927   95% CI [0.897, 0.951]
   coverage  0.909   95% CI [0.878, 0.940]
   median lead 20.0 h | 1.31 aler

**Conclusion.** Three detectors survive:
* **C09 matched signature:** 93% of its alerts are followed by a failure and it catches 91% of failures, with
  ~1.3 alerts per machine per month, ~20 h ahead.
* **C08 drift + any error:** catches 98%, but only 62% of alerts are real.
* **C10 drift on an aging part:** the least precise (41%; catches 91%), but warns **~43 h ahead**.

A random alarm would be right 4.1% of the time.

## 4 · Why these three beat the others
* **C09** pairs a part's own signal with its own error code. Each alone is weak: the signal drifts ~470 times
  per part without a failure (notebook 01, section 7), and error codes also turn up in normal operation. Real
  failures almost always show *both* for the *same* part; look-alikes almost never do.
* **C08** pairs *any* drift with *any* error, which lets more coincidences through. It is kept as the fallback
  that needs no learned part ↔ code pairing.
* **C10** doesn't wait for the error code, so it warns ~20 h earlier than C09. It is the early-warning tier.

**Why the others lose:**
* **Single signals (C01–C04)** pass every gate but are capped at 26–38% precision by the look-alike drifts.
* **Any signal drifts (C05)** catches every failure, but only 28% of its alerts are real (~6.8 per machine-month).
* **Any error code (C06)** is fairly precise (64%) but catches only 51% of failures.
* **Overdue maintenance (C07)** fails the causal check: its skill comes from the future (notebook 05, section 3).

## 5 · Should the sensor readings be smoothed?
**The line.** Every alarm compares a sensor reading to a **line** (a threshold): if the reading crosses it, the
alarm sounds. The line is set from **normal operation** only (hours far from any failure). Usually it is the
level that normal hours cross only 1% of the time, called the 99th percentile or **P99**. So in normal running
the alarm almost never sounds; before a failure, the reading drifts past the line.

**Smoothing.** Readings jitter from hour to hour, so a single odd hour can cross the line by chance. Smoothing
the reading first (for example, averaging the last 6 hours) ignores those one-hour blips. The picture below
shows a real pressure failure on a holdout machine: the hourly reading and its 6 h average, each against its
own P99 line (from all of that machine's hours, since its failure dates can't be used).

In [6]:
# One real failure: a comp3 failure (pressure is comp3's signal) on a holdout machine, the 7 days before it.
from plotly.subplots import make_subplots
pick = panel.events[panel.events.comps.map(lambda c: c == ("comp3",)) & panel.events.m.isin(hold) & (panel.events.t > 168)]
x_all = {int(m): pd.Series(panel.X["pressure"][int(m)]) for m in pick.m.unique()}
# pick the example where the 6 h average stays above its line longest in the last 48 h (the clearest picture)
def hours_above(e):
    r6 = x_all[e.m].rolling(6).mean(); line = np.nanpercentile(r6[cfg.warmup_h:], 99)
    return int((r6[e.t - 48:e.t] > line).sum())
e = pick.loc[pick.apply(hours_above, axis=1).idxmax()]
m, t = int(e.m), int(e.t)

x = x_all[m]
versions = {"hourly reading": x, "6 h average": x.rolling(6).mean()}
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=list(versions))
for row, (name, y) in enumerate(versions.items(), start=1):
    line = np.nanpercentile(y[cfg.warmup_h:], 99)  # P99 of all this holdout machine's hours (no failure dates used)
    hrs = np.arange(-168, 1)                      # hours relative to the failure (0 = failure)
    fig.add_scatter(x=hrs, y=y[t - 168:t + 1].values, row=row, col=1, showlegend=False, line=dict(color="#4c72b0"))
    fig.add_hline(y=line, row=row, col=1, line_dash="dash", line_color="red", annotation_text="P99 line")
fig.add_vline(x=0, line_dash="dot", annotation_text="failure")
fig.update_xaxes(title="hours before the failure", row=2, col=1)
fig.update_layout(height=520, title=f"Pressure on machine {panel.machines.machineID.iloc[m]} in the week before a comp3 failure")
fig.show()

**Reading it:** both versions rise above their lines in the two days before the failure. But the hourly
reading flickers back and forth across its line, so an alarm built on it would switch on **11 separate times**
that week, and each switch-on counts as an alert. The 6 h average crosses its line about 48 hours before the
failure and stays above it for most of the last two days, with a few brief dips: **5** switch-ons. Fewer, steadier alerts for the same failure means
higher precision.

**Which smoothing scores best?** To find out, we take one simple alarm, **C05: "sound the alarm when any of the
four sensors crosses its line"** (sensors only, no error codes), and build it with each kind of smoothing:

| Smoothing | What it means |
|---|---|
| none (hourly reading) | compare each hour's reading as recorded |
| 3 hours in a row | the reading must be past the line for 3 hours straight |
| 6 h average | compare the average of the last 6 hours |
| 24 h average | compare the average of the last 24 hours |
| CUSUM | a running total of how far the reading has stayed above normal |

Each version is also tried with four ways of drawing the line (one for the whole fleet, one per machine model,
one per machine, and an outlier-proof one per machine). The bars show each smoothing's **average F1 on holdout
machines** over those four (0 = useless, 1 = perfect).

In [7]:
# Every version of C05 the protocol tried. For each (smoothing, line) pair, keep the strictness that scored
# best on DEV machines, then summarise its HOLDOUT F1 per kind of smoothing.
SMOOTHING = {"raw": "none (hourly reading)", "raw k=3": "3 hours in a row", "roll6": "6 h average",
             "roll24": "24 h average", "cusum": "CUSUM"}
sw = pd.DataFrame([dict(smoothing=SMOOTHING[s["transform"] + (f" k={s['k']}" if s["k"] > 1 else "")],
                        line=s["norm"], dev_f1=s["dev"]["f1"], holdout_f1=s["hold"]["f1"])
                   for s in res["sweep"] if s["concept"] == "C05"])
best = sw.loc[sw.groupby(["smoothing", "line"]).dev_f1.idxmax()]
score = best.groupby("smoothing").holdout_f1.agg(best_score="max", average_score="mean").reindex(list(SMOOTHING.values()))
fig = px.bar(score.reset_index(), x="smoothing", y="average_score", text=score.average_score.round(2).values,
             hover_data={"best_score": ":.2f"}, range_y=[0, 1],
             labels=dict(smoothing="smoothing", average_score="F1 on holdout machines (average)"),
             title="How well the sensor-only alarm (C05) works with each kind of smoothing")
fig.show()

**Conclusion: smooth with a 6 h average.** It has the best average score and ties for the best single score
(0.47). It ignores one-hour blips but still reacts within hours to a real drift, which is why the two strongest
detectors, C09 and C08, both use it. CUSUM is the worst (0.37). The differences are small, though: no smoothing
lifts a sensor-only alarm much past 0.47. The big gain comes from adding the part's own error code (C09: 0.92).

## 6 · Testing more than one H and W: one F1 grid
10 leads H ∈ {0, 3, 6, 9, 12, 18, 24, 36, 48, 72} × 10 window lengths W ∈ {6, 12, 18, 24, 30, 36, 48, 60, 72, 96}
= 100 windows, including the one-day "within the next 24 h" (H = 0, W = 24). In every cell the windows and "normal"
are rebuilt and the leading detector (C09) is re-tuned on **dev**; the cell shows its dev F1. ~4 minutes.

In [8]:
# One dev F1 per (H, W): the best of C09's 40 variants, re-tuned for that window.
G = f1_grid(panel, bank, cfg, dev, MAPPING)

fig = px.imshow(G.values, x=[str(w) for w in G.columns], y=[str(h) for h in G.index], text_auto=".2f",
                aspect="auto", color_continuous_scale="Blues", labels=dict(x="W (hours)", y="H (hours)", color="dev F1"),
                title="Dev F1 of the leading detector for every window (H = lead, W = length)")
# Boxes: black = chosen by F1 (H=0, W=48); red = the setting we would use in practice (H=12, W=36, notebook 06).
def box(H, W, **line):
    fig.add_shape(type="rect", x0=list(G.columns).index(W) - .5, x1=list(G.columns).index(W) + .5,
                  y0=list(G.index).index(H) - .5, y1=list(G.index).index(H) + .5, line=line)
box(cfg.H, cfg.W, color="black", width=4)
box(12, 36, color="red", width=4)
fig.add_annotation(text="black = chosen by F1 (H=0, W=48) · red = practical setting (H=12, W=36)",
                   xref="paper", yref="paper", x=0, y=1.06, showarrow=False, xanchor="left")
fig.show()


**Conclusion.**
* **F1 is flat (~0.93–0.95) for every window that contains the error code logged 24 h before a failure.** Gaps
  inside this plateau (≤ 0.01) are smaller than the bootstrap noise (~±0.02).
* **F1 collapses (≤ 0.24) once the window starts 36 h or more ahead:** nothing is detectable that early.
* **Choice, by F1: H = 0, W = 48** (black box): the "within the next N hours" framing, with the smallest W
  that reaches the plateau (0.945, vs 0.935 for a one-day window). H only decides which alerts *count*; the
  detectors still alert ~20 h (C09) and ~43 h (C10) ahead.
* The **red box (H = 12, W = 36)** is what we would deploy, so every alert that counts leaves half a day to
  act. On this grid (C09 re-tuned for each window) it costs ~0.01 F1; notebook 06, section 11 has the holdout
  numbers, with and without re-tuning.

## 7 · Re-tuning the rules for the deployment window (H = 12, W = 36)
By F1 the best window is H = 0, W = 48 (section 6). But with H = 0 an alert one hour before a failure counts as
a success, which leaves no time to act. For deployment we recommend a **12-hour buffer: H = 12, W = 36**
(notebook 06, section 11 explains why 12 hours). The rules above were tuned for H = 0, W = 48, so here each kept
rule is **re-tuned for H = 12, W = 36 exactly as in the protocol** (the variant with the best F1 on dev machines),
then the rule as kept and the re-tuned rule are both scored on the holdout machines.

In [9]:
# Re-tune C09, C08 and C10 for the deployment window on DEV machines, then score both versions on HOLDOUT
# (src/evaluation.py §7, retune_for_window). ~30 s.
kept_rows = [r for r in res["concepts"] if r["status"] == "kept"]
DEP = retune_for_window(panel, cfg, bank, dev, hold, kept_rows, H=12, W=36)
pd.DataFrame([dict(rule=cid,
                   F1_at_0_48=next(r["hold"]["f1"] for r in kept_rows if r["cid"] == cid),   # as kept, own window
                   F1_as_kept_at_12_36=d["as_kept"]["f1"],                                    # same rule, 12 h buffer
                   F1_retuned_at_12_36=d["retuned"]["f1"],                                    # re-tuned for the buffer
                   precision=d["retuned"]["precision"], coverage=d["retuned"]["coverage"],
                   setting_as_kept=d["kept_variant"], setting_retuned=d["retuned_variant"])
              for cid, d in DEP["rules"].items()]).round(3)

,rule,F1_at_0_48,F1_as_kept_at_12_36,F1_retuned_at_12_36,precision,coverage,setting_as_kept,setting_retuned
0,C08,0.764,0.744,0.744,0.628,0.914,roll6·robust·p99.99·k1,roll6·robust·p99.99·k1
1,C09,0.918,0.814,0.941,0.898,0.989,roll6·machine·p99·k1,roll6·robust·p99.9·k1
2,C10,0.566,0.564,0.564,0.413,0.889,roll6·robust·p99.99·k1,roll6·robust·p99.99·k1


**Conclusion.** Only **C09's setting changes**: its line moves from the per-machine P99 to the per-machine
*outlier-proof* line at P99.9, which crosses earlier in a drift. As kept, C09 drops to F1 0.81 at H = 12,
W = 36, because 20% of failures get their first alert only in the final 12 hours. Re-tuned, it scores **0.94**
(90% precision, 99% coverage), a little higher than at H = 0, W = 48 (0.92). C08 and C10 keep their settings
(already the best on dev) and barely move (0.76 → 0.74, 0.57 → 0.56). So the 12-hour buffer costs nothing, as long
as C09 is re-tuned for it. These re-tuned rules are the ones recommended for deployment.

## 8 · The shortlist for every event type
The event is configurable (`Config(event=...)`). Sections 2–6 used "any failure". Here the same protocol runs
for each part on its own (comp1–comp4), about 1 minute each.

In [10]:
# Same protocol for each single-part event type; "any failure" is the run from section 2.
kinds = {"any": res}
for e in ("comp1", "comp2", "comp3", "comp4"):
    kinds[e] = run_event_kind(panel, cfg.with_(event=e), bank, dev, hold, n_null=cfg.n_null_shifts)

pd.DataFrame([dict(event=e, rank=k["kept"].index(c["cid"]) + 1, id=c["cid"], idea=c["title"],
                   precision=c["hold"]["precision"], coverage=c["hold"]["coverage"], F1=c["hold"]["f1"],
                   lead_h=c["lead_median"], holdout_events=k["n_hold_events"])
              for e, k in kinds.items() for c in k["concepts"] if c["status"] == "kept"]
             ).sort_values(["event", "rank"]).round(3)

  [comp1 H0_W48] 655 variants, kept ['C09', 'C10', 'C01'] (49s)


  [comp2 H0_W48] 655 variants, kept ['C09', 'C08', 'C06'] (49s)


  [comp3 H0_W48] 655 variants, kept ['C09', 'C10', 'C03'] (49s)


  [comp4 H0_W48] 655 variants, kept ['C09', 'C10', 'C04'] (49s)


,event,rank,id,idea,precision,coverage,F1,lead_h,holdout_events
1,any,1,C09,Matched signature (part's own signal + own error),0.927,0.909,0.918,20.0,350
0,any,2,C08,Drift confirmed by an error,0.628,0.977,0.764,24.0,350
2,any,3,C10,Drift on an aging part,0.413,0.903,0.566,43.0,350
4,comp1,1,C09,Matched signature (part's own signal + own error),0.891,1.000,0.942,24.0,86
5,comp1,2,C10,Drift on an aging part,0.406,0.977,0.574,40.0,86
3,comp1,3,C01,Voltage creeps up,0.255,0.721,0.376,25.0,86
8,comp2,1,C09,Matched signature (part's own signal + own error),0.909,0.985,0.945,22.0,132
7,comp2,2,C08,Drift confirmed by an error,0.694,0.985,0.814,23.0,132
6,comp2,3,C06,Any error code,0.622,0.167,0.263,24.0,132
10,comp3,1,C09,Matched signature (part's own signal + own error),0.933,0.887,0.909,21.0,62


**Conclusion.** C09 ranks first for every event type (F1 0.91–0.99). For a single part, the next places go to
that part's own evidence: drift on that part when it is old (C10), or its own signal (C01 for comp1, C03 for
comp3, C04 for comp4). comp2 is the exception: the generic error rules (C08, C06) work there because comp2 logs
two error codes.

## 9 · Save this notebook's results
Everything notebooks 05 and 06 need is saved to `outputs/04_test_and_select.json`: for every event type, each
idea's tuned rule, its dev and holdout scores with intervals, its gate results and why it was kept or rejected;
plus the part pairing, the H × W grid, the smoothing comparison and the rules re-tuned for the deployment window. Notebook 05 audits the kept detectors from
this file, and notebook 06 summarises it.

In [11]:
save_results("04_test_and_select", dict(                       # src/evaluation.py §9
    mapping=dict(MAPPING),                                        # each part's own signal and error codes (dev)
    dev_machines=(dev + 1).tolist(), hold_machines=(hold + 1).tolist(),
    # per event type; the per-variant sweep and in-memory arrays are left out (only this notebook uses them)
    kinds={e: {k: v for k, v in r.items() if k not in ("sweep", "_fires", "_mk")} for e, r in kinds.items()},
    hw_grid=G,                                                    # dev F1 per (H, W), section 6
    smoothing=score,                                              # sensor-only alarm per smoothing, section 5
    deployment=DEP,                                               # rules re-tuned for H=12, W=36, section 7
), cfg)

saved outputs/04_test_and_select.json


PosixPath('/Users/avijay/Documents/oden_precursors/outputs/04_test_and_select.json')